# Chapter 4 &mdash; The "Lasso" Shape of DFA

**Concept 12 of the Chapter 4 decomposition:** *The "Lasso" Shape of DFA*

A finite machine cannot keep sprouting states, so a long enough string must revisit one &mdash; and the loop can be repeated.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Lasso-Shape/Concept-Lasso-Shape.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


DFA look like a **lasso**: go forward a few steps, then **curl around**. Over a larger
alphabet it is not literally a lasso, but the machine still *must* curl &mdash; it cannot
go on sprouting new states forever.

So: take $w$ accepted by a DFA, at least as long as the number of states. Then some
piece of $w$ can be **repeated**. Writing $w = xyz$ with $y$ non-empty, $xy^iz$ is in
the language for **all** $i \ge 0$.

## 2. Definitions

### The $L_{3Z}$ machine again

In [ ]:
L3Z = md2mc('''DFA
IF : 0 -> S1
IF : 1 -> IF
S1 : 0 -> S2
S1 : 1 -> S1
S2 : 0 -> IF
S2 : 1 -> S2
''')
print("number of states :", len(L3Z["Q"]))

### Find the repeated state in a run

In [ ]:
def visit_trace(D, s):
    q, seen, out = D["q0"], {D["q0"]: 0}, []
    for i, ch in enumerate(s, 1):
        q = step_dfa(D, q, ch)
        if q in seen:
            out.append((seen[q], i, q))     # (first visit, second visit, state)
        else:
            seen[q] = i
    return out

## 3. Tests

A string at least as long as $|Q|$ **must** repeat a state &mdash; the pigeonhole principle.

In [ ]:
w = '0100'
print("w =", w, " |w| =", len(w), " |Q| =", len(L3Z["Q"]))
print("repeats :", visit_trace(L3Z, w)[:3])
assert visit_trace(L3Z, w), "a string this long must revisit a state"

The book's own split: $x=0$, $y=1$, $z=00$ &mdash; and $xy^iz$ stays in the language.

In [ ]:
x, y, z = '0', '1', '00'
assert accepts_dfa(L3Z, x + y + z)
for i in range(6):
    s = x + y*i + z
    print("i=%d  %-12r accepted? %s" % (i, s, accepts_dfa(L3Z, s)))
assert all(accepts_dfa(L3Z, x + y*i + z) for i in range(12))
print("\nPumping the 1-loop at S1 never leaves the language -- the DFA cannot tell.")

The machine is **forced** to admit them all: it has no way to distinguish them.

In [ ]:
print("states visited by x y^0 z :", visit_trace(L3Z, x + z))
print("states visited by x y^3 z :", visit_trace(L3Z, x + y*3 + z))
print("\nSame start, same finish -- only the loop count differs.")

## 4. Animation

Step through `0100` and watch the machine return to a state it has already visited. That revisit is the pump.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(L3Z, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Argue that a DFA over a **singleton** alphabet must literally be a lasso.
2. For a DFA $D$ recognising $L$, show there are infinitely many other DFA for $L$.
3. Find a different $x,y,z$ split of `0100` that also pumps.

In [ ]:
# Your work for the exercises above.